# Capítulo 4 — Uso de ferramentas (Tool Use)

**Livro:** começa na página **121** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py: modelos via OpenRouter, com controle de gasto
from ia import *
import json, re, time, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO)

## Automated Tool Development · PDF p. 142

#### Exercício: Validar argumentos com pydantic

Modelos às vezes mandam argumentos errados. Defina `class ArgsFrete(BaseModel)` com `cep` (8 dígitos, validado por regex) e `peso_kg` (> 0) e uma ferramenta `calcular_frete` que valida antes de executar e devolve uma mensagem de erro **útil para o modelo** quando falha.

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class ArgsFrete(BaseModel):
    cep: str = Field(pattern=r"^\d{8}$")
    peso_kg: float = Field(gt=0)
def calcular_frete(**kwargs):
    try:
        a = ArgsFrete(**kwargs)
    except ValidationError as e:
        return "ERRO de argumentos: " + "; ".join(f"{x['loc'][0]}: {x['msg']}" for x in e.errors()) + ". Corrija e tente de novo."
    return f"Frete para {a.cep}: R$ {15 + 4 * a.peso_kg:.2f}"
print(calcular_frete(cep="01310100", peso_kg=2)); print(calcular_frete(cep="01310-100", peso_kg=-1))

## Tool Use Configuration · PDF p. 145

#### Exercício: tool_choice: auto, obrigatório e nenhum

Faça a mesma pergunta ("oi, tudo bem?") com `tool_choice="auto"`, `"required"` e `"none"` e veja quando o modelo chama ferramenta.

In [ ]:
CATALOGO = {"teclado": 199.9, "mouse": 89.9, "monitor": 1299.0, "cadeira": 899.0}
PEDIDOS = {"A100": {"status": "enviado", "previsao": "2025-06-12"}, "A101": {"status": "em separação", "previsao": "2025-06-15"}}
def consultar_preco(produto: str) -> str:
    """Devolve o preço de um produto do catálogo."""
    p = CATALOGO.get(produto.lower())
    return f"{produto}: R$ {p:.2f}" if p else f"{produto} não está no catálogo"
def status_pedido(pedido_id: str) -> str:
    """Devolve o status e a previsão de entrega de um pedido."""
    p = PEDIDOS.get(pedido_id.upper())
    return json.dumps(p, ensure_ascii=False) if p else "pedido não encontrado"
FUNCOES = {"consultar_preco": consultar_preco, "status_pedido": status_pedido}
TOOLS = [{"type": "function", "function": {"name": "consultar_preco", "description": "Preço de um produto do catálogo",
          "parameters": {"type": "object", "properties": {"produto": {"type": "string"}}, "required": ["produto"]}}},
         {"type": "function", "function": {"name": "status_pedido", "description": "Status e previsão de entrega de um pedido",
          "parameters": {"type": "object", "properties": {"pedido_id": {"type": "string"}}, "required": ["pedido_id"]}}}]
for escolha in ["auto", "required", "none"]:
    r = conversar([{"role": "user", "content": "oi, tudo bem?"}], tools=TOOLS, tool_choice=escolha).choices[0].message
    print(escolha, "->", [c.function.name for c in r.tool_calls] if r.tool_calls else (r.content or "")[:60])